# Serialización reproducible: del estimador al bundle

Notebook docente de la clase 1. Partimos del módulo local de la semana 3 y hacemos explícitos los metadatos que necesita un modelo para ser ejecutable.

**Secuencia:** inspeccionar → decidir formato → escribir manifiesto → cargar con validación → romper el contrato.

## Preparación

Desde la raíz del repositorio, ejecuta uv sync. La demo usa un DummyClassifier pequeño para que nadie necesite descargar un modelo. Si existe models/wine_quality_classifier.joblib, también se puede inspeccionar el payload de la semana 3.

In [ ]:
from pathlib import Path
import json
import joblib
import sys

from sklearn.dummy import DummyClassifier

WEEK_ROOT = Path.cwd()
for candidate in (WEEK_ROOT, *WEEK_ROOT.parents):
    if (candidate / 'modules/04-model-packaging').is_dir():
        WEEK_ROOT = candidate
        break

DATA_PATH = WEEK_ROOT / 'assets/03-wine-quality/inference_samples.csv'
if not DATA_PATH.is_file():
    DATA_PATH = WEEK_ROOT.parent / 'semana3/assets/03-wine-quality/inference_samples.csv'

SOLUTION_SRC = WEEK_ROOT / 'modules/04-model-packaging/solutions/01.02-serializable-inference-module/src'
if str(SOLUTION_SRC) not in sys.path:
    sys.path.insert(0, str(SOLUTION_SRC))

from model_packaging.artifact import (
    ArtifactManifest,
    create_manifest,
    infer_wine_quality,
    load_model_bundle,
    save_model_bundle,
)
from model_packaging.contracts import WineQualityRequest
from model_packaging.preprocess import FEATURE_NAMES

WORK = WEEK_ROOT / '.tmp/semana_04_notebook'
WORK.mkdir(parents=True, exist_ok=True)
print(f'Carpeta de semana 4: {WEEK_ROOT}')
print(f'Muestras de semana 3: {DATA_PATH.exists()}')

## 1. Ver qué recibimos de la semana 3

El payload legado contiene el estimador y algunos metadatos. Antes de diseñar el nuevo formato, enumera qué decisiones siguen implícitas.

In [ ]:
estimator = DummyClassifier(strategy='constant', constant='acceptable')
estimator.fit([[0.0] * len(FEATURE_NAMES)], ['acceptable'])

legacy_payload = {
    'estimator': estimator,
    'feature_names': list(FEATURE_NAMES),
    'model_version': 'wine-quality-rf-demo-v1',
}
legacy_path = WORK / 'legacy_model.joblib'
joblib.dump(legacy_payload, legacy_path)

print(f'Claves: {list(legacy_payload)}')
print(f'Tamaño: {legacy_path.stat().st_size} bytes')
print('Faltan: schema_version, preprocessing_version, output_labels y estimator_type')

### Práctica 1 — Asignar cada dato a un formato

En parejas, completad la tabla del lienzo:

- ¿Qué debe vivir en el binario model.joblib?
- ¿Qué debe poder leer una persona en manifest.json?
- ¿Qué invariantes pertenecen al código de carga?
- ¿Qué dato de la semana 3 no debe desaparecer aunque la predicción siga funcionando?

## 2. Escribir un manifiesto mínimo

El manifiesto no sustituye al contrato de entrada. Lo completa: identifica versiones, estructura de características y etiquetas de salida.

In [ ]:
bundle_path = WORK / 'wine_quality_bundle'
manifest = create_manifest(estimator, model_version='wine-quality-rf-demo-v1')
save_model_bundle(bundle_path, estimator, manifest)

print((bundle_path / 'manifest.json').read_text(encoding='utf-8'))
print(f'Archivos del bundle: {sorted(path.name for path in bundle_path.iterdir())}')

loaded = load_model_bundle(bundle_path)
print(f'Manifiesto cargado: {loaded.manifest.model_version}')

### Práctica 2 — Anticipar fallos

Antes de ejecutar la siguiente celda, predice qué comprobación debe fallar si:

1. se invierte density y alcohol;
2. cambia preprocessing_version;
3. el estimador devuelve unknown;
4. una petición trae ph=99.

In [ ]:
invalid_manifest = json.loads((bundle_path / 'manifest.json').read_text(encoding='utf-8'))
invalid_manifest['feature_names'] = list(reversed(invalid_manifest['feature_names']))
invalid_manifest_path = WORK / 'manifest_reordered.json'
invalid_manifest_path.write_text(json.dumps(invalid_manifest), encoding='utf-8')

try:
    ArtifactManifest.model_validate_json(invalid_manifest_path.read_text(encoding='utf-8'))
except Exception as error:
    print(f'Manifiesto rechazado: {error}')

In [ ]:
sample = WineQualityRequest(
    fixed_acidity=7.4,
    volatile_acidity=0.7,
    citric_acid=0.0,
    residual_sugar=1.9,
    chlorides=0.076,
    free_sulfur_dioxide=11.0,
    total_sulfur_dioxide=34.0,
    density=0.9978,
    ph=3.51,
    sulphates=0.56,
    alcohol=9.4,
)
prediction = infer_wine_quality(loaded, sample)
print(prediction.model_dump())

try:
    WineQualityRequest.model_validate({**sample.model_dump(), 'ph': 99})
except Exception as error:
    print(f'Entrada rechazada antes del modelo: {error.errors()[0]["msg"]}')

## 3. Cierre

La práctica de la clase 2 empieza con estas decisiones. Cada pareja debe poder nombrar:

1. el archivo que contiene el estimador;
2. el archivo que permite inspeccionar la identidad del artefacto;
3. la comprobación que protege el orden de columnas;
4. la razón por la que el CSV se escribe solo después de validar todas las filas.